# 02 — Part 2: classification, comparison with Dynamic World and CGLS-LC100, diagnosis, sensitivity

**Primary method:** `Haifald/vit-eurosat` — ViT-Base/16 fine-tuned on EuroSAT RGB, applied to each of the 12 cells
(pinned to commit `MODEL_REVISION` in `config.py`). No training is done here, so there is no training/evaluation
location split to manage; the comparators and the six visual labels are never used to produce the primary labels.

**Implemented:** §0–§5 and §8 — EuroSAT inference; Dynamic World (same season and scenes) and CGLS-LC100 2019
fetched from Earth Engine and saved as GeoTIFF; crosswalks; cell majority labels; agreement with denominators; cell-level CSV.
**Still to do:** §6 (two cell diagnoses), §7 (sensitivity), §9 (report figures).

Order matters: visual labels (§1) and the sensitivity prediction (§7) are committed to git *before* the outputs they
are compared with are produced.

## 0. Setup — load config, grid, valid-data table; log new libraries

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from src.env_utils import find_repo_root, log_requirements
ROOT = find_repo_root()
import importlib, config as cfg
importlib.reload(cfg)

import numpy as np, pandas as pd, geopandas as gpd
from src import eurosat_model as EM, grid as G, gee_utils as GE, chips as CH
from src import comparators as CP, cell_agg as CA

log_requirements(["torch", "torchvision", "transformers", "affine"], cfg.REQUIREMENTS_FILE)  # absent ones skipped

cells = gpd.read_file(cfg.PATHS["geojson"] / "grid_cells.geojson")          # EPSG:4326, from notebook 01
valid = pd.read_csv(cfg.PATHS["tables"] / "cell_valid_data_part1.csv")
CELL_IDS = cells.cell_id.tolist()
assert len(CELL_IDS) == 12
CLASS_KEYS = list(cfg.CLASSES)

# Rebuild the identical UTM grid + pixel windows from config (deterministic) and check it matches the GeoJSON
cells_utm, spec = G.build_grid(*cfg.AOI_CENTER_LONLAT, cfg.GRID_ROWS, cfg.GRID_COLS,
                               cfg.CELL_SIZE_M, cfg.PIXEL_SIZE_M, cfg.CELL_ID_FMT)
assert cells_utm.cell_id.tolist() == CELL_IDS, "Grid in config no longer matches the saved GeoJSON"
print(f"Season {cfg.SEASON_TAG}: {cfg.S2_START} → {cfg.S2_END} (excl.) | DW aggregation: {cfg.DW_TEMPORAL_AGG}")

## 1. Visual labels for the six cells (hand-made, loaded read-only)
Fill `outputs/labels/visual_labels_six_cells.csv` from the chips, without AI assistance, and commit it **before**
running §2. This cell only reads it and warns if it is incomplete.

In [ ]:
labels = pd.read_csv(cfg.PATHS["labels"] / "visual_labels_six_cells.csv", dtype=str).fillna("")
missing = labels[labels.visual_label.eq("") | labels.visual_cue.eq("") | labels.confidence.eq("")]
if len(missing):
    print(f"WARNING: {len(missing)} of {len(labels)} visual labels incomplete:", missing.cell_id.tolist())
bad = set(labels.visual_label) - set(CLASS_KEYS) - {cfg.MIXED_LABEL, ""}
assert not bad, f"Labels outside the class scheme: {bad}"
labels

## 2. Primary classification — EuroSAT ViT on all 12 cells

### 2a. Model and its training data (disclosure)
* Base: `google/vit-base-patch16-224` (ImageNet-21k pre-training, ImageNet fine-tuning), fine-tuned on `cm93/eurosat`.
* EuroSAT: 27,000 RGB patches, 64 × 64 px at 10 m, **Sentinel-2A, Level-1C (no atmospheric correction), 34 European countries**,
  10 classes: AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial, Pasture, PermanentCrop, Residential, River, SeaLake.
* **Overlap with comparators:** none known. EuroSAT contains no Indian locations, and neither Dynamic World nor CGLS-LC100
  labels were used to build it. The AOI is therefore fully out of the model's training geography — a domain-shift risk,
  not a leakage risk.
* **Mismatches we cannot remove:** EuroSAT patches are single cloud-free scenes and 640 m wide; our inputs are a seasonal
  median composite and 500 m cells. EuroSAT has **no bare-soil class**, so `bare` can never be predicted.
* The RGB stretch EuroSAT used to make its JPEGs is not documented; our fixed stretch (`RGB_STRETCH_*`) is an assumption.

In [ ]:
clf = EM.load_classifier(cfg.MODEL_ID, cfg.MODEL_REVISION, cfg.MODEL_DEVICE)
assert sorted(clf.model.config.id2label.values()) == sorted(cfg.EUROSAT_CLASSES)
proc = clf.image_processor
print("Processor:", type(proc).__name__, proc.size, "resample", proc.resample,
      "| mean", proc.image_mean, "std", proc.image_std)

### 2b. Prepare inputs: native 50 × 50 → 64 × 64
Each cell's native chip (`outputs/chips/rgb_native/`, one pixel = 10 m) is resized **bilinearly to 64 × 64**, the EuroSAT
patch size. The model's own processor then resizes 64 → 224. The exact 64 × 64 image fed to the model is saved to
`outputs/model_inputs/` so every prediction can be traced to its input.

In [ ]:
inputs = {cid: EM.prepare_input(cfg.PATHS["chips_native"] / f"{cid}.png",
                                cfg.PATHS["model_inputs"] / f"{cid}.png",
                                cfg.MODEL_INPUT_PX, cfg.MODEL_INPUT_RESAMPLE)
          for cid in CELL_IDS}
print({k: v.size for k, v in list(inputs.items())[:3]}, "...")

### 2c. Run inference; save raw responses
One unedited JSON per cell in `outputs/model_responses/raw/`, with the full 10-class probability vector and the run settings.

In [ ]:
run_info = {"model_id": cfg.MODEL_ID, "model_revision": cfg.MODEL_REVISION,
            "input_image": "outputs/model_inputs/{cell}.png",
            "input_px": cfg.MODEL_INPUT_PX, "input_resample": cfg.MODEL_INPUT_RESAMPLE,
            "source_chip_px": cfg.CELL_SIZE_M // cfg.PIXEL_SIZE_M,
            "imagery": f"{cfg.S2_COLLECTION} {cfg.S2_PLATFORMS} {cfg.COMPOSITE_METHOD} {cfg.S2_START}..{cfg.S2_END}",
            "rgb_stretch_dn": [cfg.RGB_STRETCH_MIN, cfg.RGB_STRETCH_MAX], "rgb_gamma": cfg.RGB_GAMMA}
probs = EM.classify_cells(clf, inputs, cfg.PATHS["model_raw"], cfg.MODEL_TOP_K, run_info)
probs.round(3)

### 2d. Aggregate to the project classes
**Rule:** EuroSAT probabilities are **summed** into the project classes via `EUROSAT_CROSSWALK`
(e.g. AnnualCrop + PermanentCrop → cropland); the cell label is the class with the highest summed probability.
Summing (rather than taking the top EuroSAT class) avoids splitting one land cover across sibling classes
— e.g. Pasture 0.35 + HerbaceousVegetation 0.30 is 0.65 shrub_grass even if AnnualCrop is the single top class at 0.35.
Uncertainty is reported three ways: max probability, margin to the runner-up, and normalised entropy;
cells with max probability < `MODEL_UNCERTAINTY_THRESHOLD` are flagged, but their label is kept.

In [ ]:
agg = EM.aggregate_to_classes(probs, cfg.EUROSAT_CROSSWALK, CLASS_KEYS, cfg.MODEL_UNCERTAINTY_THRESHOLD)
baseline = (agg.merge(probs, on="cell_id")
               .merge(valid[["cell_id", "n_valid_pixels", "valid_frac", "obs_clear_median"]], on="cell_id"))
out = cfg.PATHS["runs_baseline"] / "eurosat_cell_predictions.csv"
if out.exists():
    print("Baseline already exists — NOT overwritten (retain original outputs):", out.relative_to(ROOT))
else:
    baseline.to_csv(out, index=False)
    print("Saved", out.relative_to(ROOT))
baseline[["cell_id", "method_label", "method_top_eurosat", "method_max_prob",
          "method_margin", "method_entropy", "method_uncertain", "valid_frac"]]

### 2e. Figure: chip, label and top-3 EuroSAT classes per cell

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
eu_cols = [c for c in probs.columns if c.startswith("p_")]
fig, axes = plt.subplots(cfg.GRID_ROWS, cfg.GRID_COLS, figsize=(cfg.GRID_COLS * 2.6, cfg.GRID_ROWS * 3.1))
for ax, cid in zip(axes.ravel(), CELL_IDS):
    r = baseline.set_index("cell_id").loc[cid]
    top3 = probs.set_index("cell_id").loc[cid, eu_cols].astype(float).nlargest(3)
    ax.imshow(Image.open(cfg.PATHS["chips_rgb"] / f"{cid}.png"))
    ax.set_title(f"{cid}: {r.method_label} ({r.method_max_prob:.2f}){' ?' if r.method_uncertain else ''}", fontsize=9)
    ax.set_xlabel("\n".join(f"{k[2:]} {v:.2f}" for k, v in top3.items()), fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"EuroSAT ViT on {cfg.CELL_SIZE_M} m cells (50→{cfg.MODEL_INPUT_PX} px) — '?' = max prob < {cfg.MODEL_UNCERTAINTY_THRESHOLD}", fontsize=10)
fig.tight_layout()
fig.savefig(cfg.PATHS["figures"] / "eurosat_predictions_panel.png", dpi=cfg.PNG_DPI)
plt.show()

## 3. Comparator 1 — Dynamic World V1, same season, same scenes, same footprint

| Choice | Setting |
|---|---|
| Product | `GOOGLE/DYNAMICWORLD/V1` — one 10 m prediction per Sentinel-2 L1C scene; 9 class probabilities + `label` |
| Window | `S2_START`–`S2_END` (season from `SEASON_START_MONTH`/`SEASON_END_MONTH`), identical to the S2 composite |
| Scenes | `DW_MATCH_S2_SCENES`: only DW images whose `system:index` (= source L1C scene ID) is in the S2 scene inventory, i.e. the same Sentinel-2A scenes, ≤35 % scene cloud |
| Cloud mask | DW's own cloud/shadow mask **plus** the same Cloud Score+ mask as the composite (`DW_APPLY_CLOUD_SCORE`) |
| Temporal aggregation | `DW_TEMPORAL_AGG` = **mode** (default): per pixel, the share of valid observations in each class; label = most frequent class. `mean` / `median` aggregate the probability bands instead — switch in `config.py` |
| Resolution | Native 10 m; fetched on the analysis grid (same UTM lattice) with nearest-neighbour |
| Cell label | Crosswalk each pixel's aggregated DW class (`DW_CROSSWALK`), then **majority rule** over the cell's valid pixels: plurality class = `dw_label`; `dw_label_strict` = that class only if it covers ≥ `DOMINANT_CLASS_THRESHOLD` of valid pixels, else `mixed` |
| Valid data | pixels with ≥1 valid DW observation; `n_obs` per pixel is kept |

In [ ]:
GE.init_ee(cfg.EE_PROJECT)
aoi_ee = GE.aoi_rectangle(spec)

s2_inv = pd.read_csv(cfg.S2_INVENTORY_CSV)
scene_ids = s2_inv.system_index.tolist() if cfg.DW_MATCH_S2_SCENES else None
dw = CP.dw_collection(aoi_ee, cfg.S2_START, cfg.S2_END, cfg.DW_COLLECTION, scene_ids,
                      cfg.CLOUD_SCORE_COLLECTION if cfg.DW_APPLY_CLOUD_SCORE else None,
                      cfg.CS_BAND, cfg.CS_THRESHOLD)
dw_inv = CP.dw_inventory(dw)
assert len(dw_inv), "No Dynamic World images matched the S2 scenes — check the season and scene IDs"
dw_inv.to_csv(cfg.DW_INVENTORY_CSV, index=False)

s2_inv["has_dw"] = s2_inv.system_index.isin(set(dw_inv.system_index))
print(f"S2 scenes in season: {len(s2_inv)} | DW images matched: {len(dw_inv)} | "
      f"S2 scenes with no DW image: {(~s2_inv.has_dw).sum()} {s2_inv.loc[~s2_inv.has_dw, 'system_index'].tolist()}")
dw_inv.head()

In [ ]:
dw_img = CP.dw_seasonal(dw, cfg.DW_CLASS_NAMES, cfg.DW_TEMPORAL_AGG)
dw_arr, dw_bands = GE.fetch_grid_array(dw_img, spec, cfg.NODATA)
CH.write_geotiff(dw_arr, dw_bands, spec, cfg.DW_TIF, cfg.NODATA)
db = {b: i for i, b in enumerate(dw_bands)}
print("Saved", cfg.DW_TIF.relative_to(ROOT), dw_arr.shape, dw_bands)

In [ ]:
# Pixel crosswalk -> cell majority
dw_code_map = {i: cfg.DW_CROSSWALK[n] for i, n in enumerate(cfg.DW_CLASS_NAMES)}
dw_cls = CA.crosswalk_codes(dw_arr[..., db["label_agg"]], dw_code_map, CLASS_KEYS, cfg.NODATA)
dw_cells = CA.cell_majority(dw_cls, cells_utm, CLASS_KEYS, cfg.DOMINANT_CLASS_THRESHOLD, "dw", cfg.MIXED_LABEL)

# Uncertainty / support: crosswalked class support averaged over the cell, per-pixel confidence, n_obs
dw_sup = CA.crosswalk_support(dw_arr[..., [db[f"sup_{n}"] for n in cfg.DW_CLASS_NAMES]],
                              cfg.DW_CLASS_NAMES, cfg.DW_CROSSWALK, CLASS_KEYS, cfg.NODATA)
for extra in [CA.cell_mean_support(dw_sup, cells_utm, CLASS_KEYS, "dw"),
              CA.cell_band_stats(dw_arr, cells_utm, db["n_obs"], cfg.NODATA, "dw_n_obs"),
              CA.cell_band_stats(dw_arr, cells_utm, db["conf_agg"], cfg.NODATA, "dw_pixel_conf", ("mean", "median"))]:
    dw_cells = dw_cells.merge(extra, on="cell_id")
dw_cells.to_csv(cfg.DW_CELLS_CSV, index=False)
dw_cells[["cell_id", "dw_label", "dw_top_frac", "dw_second", "dw_label_strict",
          "dw_n_valid_px", "dw_n_px", "dw_n_obs_median", "dw_pixel_conf_mean"]]

## 4. Comparator 2 — Copernicus CGLS-LC100 Collection 3, 2019 epoch

| Choice | Setting |
|---|---|
| Product | `COPERNICUS/Landcover/100m/Proba-V-C3/Global/2019`, band `discrete_classification` (+ `discrete_classification-proba`, `data-density-indicator`, cover fractions) |
| Temporal definition | **Annual 2019 map** from the PROBA-V 100 m time series — it cannot be restricted to our season. This mismatch is reported, not removed |
| Resolution | Native 100 m (geographic grid, not aligned with our UTM grid). Saved natively **and** resampled to the 10 m analysis grid with **nearest neighbour**, so every 10 m pixel carries the 100 m class it falls in |
| Cell label | Crosswalk (`CGLS_CROSSWALK`) per 10 m pixel, then the same **majority rule** — equivalent to an area-weighted majority of the 100 m pixels overlapping the cell. A CGLS pixel is ~0.00099° (≈108 m at 13° N), so only ~16–25 native pixel centres fall inside a 500 m cell; `cgls_native_px_centres` records the exact count per cell — the effective sample is small |
| Valid data | pixels with a mapped class; code 0 (unknown) is counted as unmapped |

In [ ]:
cg_img, cg_bands_used = CP.cgls_image(cfg.CGLS_IMAGE, cfg.CGLS_EXPORT_BANDS)

# (a) on the 10 m analysis grid, nearest neighbour
cg_arr, cg_bands = GE.fetch_grid_array(cg_img, spec, cfg.NODATA)
CH.write_geotiff(cg_arr, cg_bands, spec, cfg.CGLS_TIF_10M, cfg.NODATA)

# (b) on its own native 100 m grid, covering the AOI
cg_nat, nat_bands, nat_tf, nat_crs = CP.fetch_native_array(cg_img, tuple(cells.total_bounds), cfg.NODATA)
CP.write_geotiff_affine(cg_nat, nat_bands, nat_tf, nat_crs, cfg.CGLS_TIF_NATIVE, cfg.NODATA)
print("Saved", cfg.CGLS_TIF_10M.relative_to(ROOT), cg_arr.shape, "|",
      cfg.CGLS_TIF_NATIVE.relative_to(ROOT), cg_nat.shape, nat_crs, f"pixel {nat_tf.a:.6f}°")

In [ ]:
cb = {b: i for i, b in enumerate(cg_bands)}
cg_codes = cg_arr[..., cb[cfg.CGLS_BAND]]
cg_cls = CA.crosswalk_codes(cg_codes, cfg.CGLS_CROSSWALK, CLASS_KEYS, cfg.NODATA)
cg_cells = CA.cell_majority(cg_cls, cells_utm, CLASS_KEYS, cfg.DOMINANT_CLASS_THRESHOLD, "cgls", cfg.MIXED_LABEL)

# Most common ORIGINAL CGLS code per cell (for tracing the crosswalk), quality bands, native pixel count
cg_cells["cgls_top_code"] = [int(pd.Series(cg_codes[c.px_row0:c.px_row1, c.px_col0:c.px_col1].ravel()).mode()[0])
                             for _, c in cells_utm.iterrows()]
for band, name in [("discrete_classification-proba", "cgls_proba"), ("data-density-indicator", "cgls_density")]:
    if band in cb:
        cg_cells = cg_cells.merge(CA.cell_band_stats(cg_arr, cells_utm, cb[band], cfg.NODATA, name, ("median", "min")), on="cell_id")

H, W = cg_nat.shape[:2]
cc, rr = np.meshgrid(np.arange(W) + 0.5, np.arange(H) + 0.5)
xs, ys = nat_tf * (cc.ravel(), rr.ravel())
cg_cells["cgls_native_px_centres"] = cg_cells.cell_id.map(CA.count_points_in_cells(xs, ys, cells.to_crs(nat_crs)))
cg_cells.to_csv(cfg.CGLS_CELLS_CSV, index=False)
cg_cells[["cell_id", "cgls_label", "cgls_top_frac", "cgls_second", "cgls_label_strict", "cgls_top_code",
          "cgls_n_valid_px", "cgls_native_px_centres"] + [c for c in cg_cells if c.startswith(("cgls_proba", "cgls_density"))]]

## 5. Crosswalk, aggregation rules and agreement (with denominators)
* **Crosswalks** (`config.py`): EuroSAT 10 → 6, Dynamic World 9 → 6, CGLS discrete codes → 6 project classes; saved to `outputs/tables/crosswalk.csv`.
* **Cell rules:** EuroSAT — argmax of summed crosswalked probabilities (one prediction per cell).
  Dynamic World and CGLS — per-pixel crosswalk, then majority rule over valid pixels in the cell (ties → first class in `CLASSES` order).
* **Agreement ≠ accuracy.** Agreement between two products is reported separately from the check against the six visual
  labels, which are themselves an imperfect, very small reference (n ≤ 6). Every percentage carries its denominator;
  a cell is excluded from a comparison only if one of the two labels is missing.

In [ ]:
xw = pd.DataFrame(
    [{"product": "EuroSAT ViT", "source_class": k, "project_class": v} for k, v in cfg.EUROSAT_CROSSWALK.items()]
  + [{"product": "Dynamic World V1", "source_class": k, "project_class": v} for k, v in cfg.DW_CROSSWALK.items()]
  + [{"product": "CGLS-LC100 C3", "source_class": k, "project_class": v} for k, v in cfg.CGLS_CROSSWALK.items()])
xw["project_class"] = xw.project_class.fillna("(unmapped)")
xw.to_csv(cfg.PATHS["tables"] / "crosswalk.csv", index=False)

eu = pd.read_csv(cfg.PATHS["runs_baseline"] / "eurosat_cell_predictions.csv")
merged = (cells[["cell_id", "row", "col", "centroid_lon", "centroid_lat"]]
          .merge(labels, on="cell_id", how="left")
          .merge(eu, on="cell_id").merge(dw_cells, on="cell_id").merge(cg_cells, on="cell_id"))

agree_cmp = CA.pairwise_agreement(merged, [("method_label", "dw_label"), ("method_label", "cgls_label"),
                                           ("dw_label", "cgls_label")],
                                  "agreement between products (not accuracy)")
vis = merged[merged.visual_label.fillna("").ne("") & merged.visual_label.ne(cfg.MIXED_LABEL)]
agree_vis = CA.pairwise_agreement(vis, [("visual_label", "method_label"), ("visual_label", "dw_label"),
                                        ("visual_label", "cgls_label")],
                                  "check vs visual labels (non-mixed only; imperfect, tiny)")
agreement = pd.concat([agree_cmp, agree_vis], ignore_index=True)
agreement.to_csv(cfg.AGREEMENT_CSV, index=False)
print("Visual labels marked mixed (not scored):",
      merged.loc[merged.visual_label.eq(cfg.MIXED_LABEL), "cell_id"].tolist())
agreement

### 5b. Map panel: composite, Dynamic World, CGLS (crosswalked) and EuroSAT cell labels

In [ ]:
import rasterio
with rasterio.open(cfg.S2_COMPOSITE_TIF) as src:
    s2 = np.dstack([src.read(list(src.descriptions).index(b) + 1) for b in cfg.RGB_BANDS])
rgb8, _ = CH.stretch_rgb(s2, cfg.RGB_STRETCH_MIN, cfg.RGB_STRETCH_MAX, cfg.RGB_GAMMA, cfg.NODATA)
CH.comparison_panel(rgb8, {f"Dynamic World ({cfg.DW_TEMPORAL_AGG}, {cfg.SEASON_TAG})": (dw_cls, merged.set_index("cell_id").dw_label),
                           f"CGLS-LC100 C3 {cfg.YEAR} (100 m → 10 m nearest)": (cg_cls, merged.set_index("cell_id").cgls_label)},
                    merged.set_index("cell_id").method_label, cells_utm, spec, CLASS_KEYS, cfg.CLASS_COLORS,
                    cfg.PATHS["figures"] / f"comparison_panel_{cfg.SEASON_TAG}_{cfg.DW_TEMPORAL_AGG}.png", cfg.PNG_DPI)
from IPython.display import Image as IPImage, display
display(IPImage(cfg.PATHS["figures"] / f"comparison_panel_{cfg.SEASON_TAG}_{cfg.DW_TEMPORAL_AGG}.png", width=900))

## 6. Two disagreement / uncertain cells — images, outputs, observed vs hypothesised cause

## 7. Sensitivity test — prediction committed first (`docs/sensitivity_prediction.md`), then rerun to `outputs/runs/sensitivity/`

## 8. Cell-level CSV → `outputs/tables/cell_level_results.csv`
One row per cell (all 12, none dropped): visual label, cue and confidence where available; EuroSAT label and uncertainty;
DW and CGLS labels with class shares, valid-pixel counts and quality measures; pairwise agreement flags; and the paths
needed to trace the row back to its chip, model input and raw response.

In [ ]:
ag = lambda a, b: np.where(merged[a].fillna("").eq("") | merged[b].fillna("").eq(""), None, merged[a] == merged[b])
results = merged.assign(
    agree_eurosat_dw=ag("method_label", "dw_label"),
    agree_eurosat_cgls=ag("method_label", "cgls_label"),
    agree_dw_cgls=ag("dw_label", "cgls_label"),
    chip_path="outputs/chips/rgb/" + merged.cell_id + ".png",
    model_input_path="outputs/model_inputs/" + merged.cell_id + ".png",
    raw_response_path="outputs/model_responses/raw/" + merged.cell_id + ".json",
    season=cfg.SEASON_TAG, dw_aggregation=cfg.DW_TEMPORAL_AGG,
)
cols = (["cell_id", "row", "col", "centroid_lon", "centroid_lat",
         "visual_label", "secondary_label", "visual_cue", "confidence",
         "n_valid_pixels", "valid_frac", "obs_clear_median",
         "method_label", "method_top_eurosat", "method_max_prob", "method_margin", "method_entropy", "method_uncertain",
         "dw_label", "dw_top_frac", "dw_second", "dw_second_frac", "dw_label_strict",
         "dw_n_px", "dw_n_valid_px", "dw_n_nodata_px", "dw_n_obs_median", "dw_n_obs_min", "dw_pixel_conf_mean",
         "cgls_label", "cgls_top_frac", "cgls_second", "cgls_second_frac", "cgls_label_strict", "cgls_top_code",
         "cgls_n_px", "cgls_n_valid_px", "cgls_n_unmapped_px", "cgls_native_px_centres"]
        + [c for c in merged if c.startswith(("cgls_proba", "cgls_density"))]
        + ["agree_eurosat_dw", "agree_eurosat_cgls", "agree_dw_cgls",
           "season", "dw_aggregation", "chip_path", "model_input_path", "raw_response_path"])
results = results.rename(columns={"confidence": "visual_confidence"})
cols = [("visual_confidence" if c == "confidence" else c) for c in cols]
results[[c for c in cols if c in results]].to_csv(cfg.CELL_RESULTS_CSV, index=False)
print("Saved", cfg.CELL_RESULTS_CSV.relative_to(ROOT), "|", len(results), "rows")
results[["cell_id", "visual_label", "method_label", "dw_label", "cgls_label",
         "agree_eurosat_dw", "agree_eurosat_cgls", "agree_dw_cgls"]]

In [ ]:
# Run metadata for Part 2 (products, windows, scene counts, rules)
meta2 = {
    "season": {"tag": cfg.SEASON_TAG, "start": cfg.S2_START, "end_exclusive": cfg.S2_END},
    "eurosat": {"model": cfg.MODEL_ID, "revision": cfg.MODEL_REVISION, "input_px": cfg.MODEL_INPUT_PX,
                "cell_rule": "argmax of summed crosswalked probabilities"},
    "dynamic_world": {"collection": cfg.DW_COLLECTION, "n_images": len(dw_inv), "n_s2_scenes": len(s2_inv),
                      "matched_to_s2_scenes": cfg.DW_MATCH_S2_SCENES, "cloud_score_mask": cfg.DW_APPLY_CLOUD_SCORE,
                      "temporal_aggregation": cfg.DW_TEMPORAL_AGG, "resampling": "nearest, native 10 m UTM",
                      "cell_rule": f"pixel crosswalk + majority; strict if >= {cfg.DOMINANT_CLASS_THRESHOLD}"},
    "cgls": {"image": cfg.CGLS_IMAGE, "bands": cg_bands_used, "temporal_definition": "annual 2019 epoch",
             "native_crs": nat_crs, "native_pixel_deg": nat_tf.a,
             "resampling": "nearest to 10 m analysis grid", "cell_rule": "pixel crosswalk + majority (area-weighted)"},
    "files": {k: str(v.relative_to(ROOT)) for k, v in {
        "dw_tif": cfg.DW_TIF, "cgls_10m_tif": cfg.CGLS_TIF_10M, "cgls_native_tif": cfg.CGLS_TIF_NATIVE,
        "agreement": cfg.AGREEMENT_CSV, "cell_results": cfg.CELL_RESULTS_CSV}.items()},
}
_ = (cfg.PATHS["metadata"] / f"run_metadata_part2_{cfg.SEASON_TAG}_{cfg.DW_TEMPORAL_AGG}.json").write_text(
    json.dumps(meta2, indent=2, default=str))

## 9. Figures for the report